In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ----------------------------------
# 1. 데이터 로드 및 정렬
# ----------------------------------

df = pd.read_csv('extracted_data.csv')
df = df.sort_values(by=['id', 'year']).reset_index(drop=True)

print(f'전체 행: {len(df):,}  |  고유 고객: {df["id"].nunique():,}')

# ----------------------------------
# 2. 파생변수 생성
# ----------------------------------

df['prev_provider']    = df.groupby('id')['provider'].shift(1)
df['provider_changed'] = (df['provider'] != df['prev_provider']).astype(int)
df['total_changes']    = df.groupby('id')['provider_changed'].cumsum()
df['prev_cost']        = df.groupby('id')['monthly_total_cost'].shift(1)
df['cost_change_rate'] = (df['monthly_total_cost'] - df['prev_cost']) / (df['prev_cost'] + 1)
df['tenure']           = df.groupby('id').cumcount() + 1

# ----------------------------------
# 3. next_provider 생성
# ----------------------------------

df['next_provider'] = df.groupby('id')['provider'].shift(-1)

provider_map = {1.0: 'SKT', 2.0: 'KT', 3.0: 'LGU+'}

# ----------------------------------
# 4. 실제 전환 이탈 고객 추출 (알뜰폰 제외)
# ----------------------------------

switching_df = df[
    (df['churn'] == 1.0) &
    (df['next_provider'].notna()) &
    (df['provider'].notna()) &
    (df['provider'].isin([1.0, 2.0, 3.0])) &
    (df['next_provider'].isin([1.0, 2.0, 3.0])) &
    (df['provider'] != df['next_provider'])
].copy()

print(f'\n대상 고객 수: {len(switching_df):,}명')
print(switching_df['next_provider'].map(provider_map).value_counts())

# ----------------------------------
# 5. 전처리
# ----------------------------------

switching_df['monthly_installment'] = switching_df['monthly_installment'].fillna(0)
switching_df['monthly_total_cost']  = switching_df['monthly_total_cost'].fillna(switching_df['monthly_total_cost'].median())
switching_df['is_mobile_bundled']   = switching_df['is_mobile_bundled'].fillna(0)
switching_df['household_size']      = switching_df['household_size'].fillna(switching_df['household_size'].median())
switching_df['prev_cost']           = switching_df['prev_cost'].fillna(switching_df['monthly_total_cost'])
switching_df['cost_change_rate']    = switching_df['cost_change_rate'].fillna(0)
switching_df['provider_changed']    = switching_df['provider_changed'].fillna(0)
switching_df['total_changes']       = switching_df['total_changes'].fillna(0)

features = [
    'age', 'gender', 'income', 'school', 'area', 'job', 'marriage',
    'monthly_total_cost', 'monthly_installment', 'cost_payer',
    'provider', 'household_size', 'is_mobile_bundled',
    'provider_changed', 'total_changes', 'cost_change_rate', 'tenure'
]

X = switching_df[features]
y = switching_df['next_provider'].astype(int)

le = LabelEncoder()
y_encoded    = le.fit_transform(y)
target_names = [provider_map[float(cls)] for cls in le.classes_]

print(f'\n피처 수: {len(features)}개')
print(f'클래스: {target_names}')

# ----------------------------------
# 6. 학습 / 검증 분할 (8:2)
# ----------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f'\nTrain: {len(X_train):,}  |  Val: {len(X_val):,}')

# ----------------------------------
# 7. 클래스 가중치 계산
# ----------------------------------

# XGBoost는 class_weight 파라미터가 없어서 sample_weight로 처리
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# ----------------------------------
# 8. XGBoost 모델 학습
# ----------------------------------

model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=0.5,
    random_state=42,
    verbosity=0,
    early_stopping_rounds=30
)

model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print('train finish')

# ----------------------------------
# 9. 평가
# ----------------------------------

y_val_pred = model.predict(X_val)
y_val_prob = model.predict_proba(X_val)

print('========================================================')
print(f' [모델 평가] 검증 데이터셋 정확도: {accuracy_score(y_val, y_val_pred):.2%}')
print('========================================================')

print('\n[Classification Report]')
print(classification_report(y_val, y_val_pred, target_names=target_names))

print('[Confusion Matrix]')
cm = confusion_matrix(y_val, y_val_pred)
cm_df = pd.DataFrame(
    cm,
    index=[f'실제_{t}'   for t in target_names],
    columns=[f'예측_{t}' for t in target_names]
)
print(cm_df)

# ----------------------------------
# 10. 변수 중요도
# ----------------------------------

xgb_importance_df = pd.DataFrame({
    'feature'   : features,
    'xgb_importance': model.feature_importances_
}).sort_values('xgb_importance', ascending=False).reset_index(drop=True)

print('\n[변수 중요도 Top 17]')
print(xgb_importance_df.to_string(index=False))

# ----------------------------------
# 11. 최종 결과 출력
# ----------------------------------

pred_probs       = model.predict_proba(X)
pred_indices     = model.predict(X)
max_probs        = np.max(pred_probs, axis=1)

switching_df['현재 통신사']      = switching_df['provider'].map(provider_map)
switching_df['실제 다음 통신사'] = switching_df['next_provider'].map(provider_map)
switching_df['예측 다음 통신사'] = pd.Series(le.inverse_transform(pred_indices), index=switching_df.index).map(provider_map)
switching_df['이동 확률(%)']     = [f'{p*100:.1f}%' for p in max_probs]

for i, name in enumerate(target_names):
    switching_df[f'{name} 확률(%)'] = [f'{p[i]*100:.1f}%' for p in pred_probs]

output_cols = (
    ['id', 'year', '현재 통신사'] +
    [f'{t} 확률(%)' for t in target_names] +
    ['실제 다음 통신사']
)

result_show = switching_df[output_cols].head(10).copy()
result_show.columns = (
    ['고객 ID', '이탈 연도', '현재 통신사'] +
    [f'{t} 확률(%)' for t in target_names] +
    ['실제 다음 통신사']
)

print('\n========== 이탈 고객별 차기 통신사 예측 결과 (상위 10개) ==========')
print(result_show.to_string(index=False))

In [ ]:
import os
import joblib

# 1. 'models' 폴더가 없으면 자동 생성
model_dir = './models'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
    print(f"'{model_dir}' 폴더를 생성했습니다.")

# 2. 파일 저장 경로 설정 (.joblib 확장자 사용)
xgb_model_path = os.path.join(model_dir, 'next_xgb_churn_v3.joblib')
le_path        = os.path.join(model_dir, 'label_encoder.joblib')

# 3. 모델 및 레이블 인코더 저장
joblib.dump(model, xgb_model_path)
joblib.dump(le, le_path)

print('========================================================')
print(f' 모델 저장 완료: {xgb_model_path}')
print(f' 인코더 저장 완료: {le_path}')
print('========================================================')